# 🏋️ Notebook 4 — Model Training
**Pill Counter | Computer Vision Pipeline**

> **Goal:** Train a YOLOv8 model on the prepared pill dataset, track metrics per epoch, compare model variants, and save the best checkpoint.

---
**Sections:**
1. Imports & config
2. Dataset validation
3. Model size comparison (nano vs small vs medium)
4. Training run with live metric tracking
5. Training curve visualisation
6. Checkpoint management


## 4.1 — Imports & Config

In [ ]:
import torch, yaml, json, time, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

plt.rcParams["figure.dpi"] = 120

with open("config.yaml") as f:
    CFG = yaml.safe_load(f)

DATASET_YAML = Path(CFG["paths"]["processed_data"]) / "dataset.yaml"
OUTPUT_DIR   = Path(CFG["paths"]["trained_models"])
IMG_SIZE     = CFG["dataset"]["img_size"]

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device       : {DEVICE}")
print(f"Dataset YAML : {DATASET_YAML}")
print(f"Output dir   : {OUTPUT_DIR}")


## 4.2 — Dataset Validation

In [ ]:
assert DATASET_YAML.exists(), f"dataset.yaml not found at {DATASET_YAML} — run Notebook 2 first."

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

print("dataset.yaml contents:")
for k, v in ds_cfg.items():
    print(f"  {k}: {v}")

# Count images per split
ds_root = Path(ds_cfg["path"])
for split in ["train", "val", "test"]:
    img_dir = ds_root / ds_cfg[split]
    n = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    print(f"  {split}: {n} images")

print("\n✅  Dataset config is valid.")


## 4.3 — Model Variant Comparison (Parameter Count)

> Before committing to a long training run, inspect each model's parameter count and theoretical FLOPs to match hardware constraints.

In [ ]:
variants   = ["yolov8n", "yolov8s", "yolov8m", "yolov8l"]
param_data = []

print(f"{'Variant':>10}  {'Params (M)':>12}  {'Size (MB)':>10}")
print("-" * 38)

for v in variants:
    try:
        m = YOLO(f"{v}.pt")
        params = sum(p.numel() for p in m.model.parameters()) / 1e6
        size_mb = sum(p.numel() * p.element_size() for p in m.model.parameters()) / 1e6
        param_data.append((v, params, size_mb))
        marker = " ← selected" if v == CFG["model"]["name"] else ""
        print(f"{v:>10}  {params:>10.1f}M  {size_mb:>8.1f} MB{marker}")
    except Exception as e:
        print(f"{v:>10}  ERROR: {e}")

# Bar chart
if param_data:
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [d[0] for d in param_data]
    params = [d[1] for d in param_data]
    colors = ["#F5844C" if l == CFG["model"]["name"] else "#4C8BF5" for l in labels]
    ax.bar(labels, params, color=colors)
    ax.set_ylabel("Parameters (M)")
    ax.set_title("YOLOv8 Variant Comparison")
    for i, (l, p) in enumerate(zip(labels, params)):
        ax.text(i, p+0.2, f"{p:.1f}M", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("results/visualizations/model_comparison.png", bbox_inches="tight")
    plt.show()


## 4.4 — Training Configuration

In [ ]:
# ── Centralise ALL training hyperparameters here ──
# Modify this cell to run ablation experiments

TRAIN_CFG = {
    "data"          : str(DATASET_YAML),
    "epochs"        : CFG["training"]["epochs"],
    "imgsz"         : IMG_SIZE,
    "batch"         : CFG["training"]["batch_size"],
    "device"        : DEVICE,
    "lr0"           : CFG["training"]["learning_rate"],
    "patience"      : CFG["training"]["patience"],
    "momentum"      : CFG["training"]["momentum"],
    "weight_decay"  : CFG["training"]["weight_decay"],
    "warmup_epochs" : CFG["training"]["warmup_epochs"],
    "optimizer"     : CFG["training"]["optimizer"],
    "project"       : str(OUTPUT_DIR),
    "name"          : "pill_detector",
    "save"          : True,
    "val"           : True,
    "verbose"       : True,
    "seed"          : 42,
    # Augmentation
    "hsv_h"    : CFG["augmentation"]["hsv_h"],
    "hsv_s"    : CFG["augmentation"]["hsv_s"],
    "hsv_v"    : CFG["augmentation"]["hsv_v"],
    "degrees"  : CFG["augmentation"]["degrees"],
    "translate": CFG["augmentation"]["translate"],
    "scale"    : CFG["augmentation"]["scale"],
    "flipud"   : CFG["augmentation"]["flipud"],
    "fliplr"   : CFG["augmentation"]["fliplr"],
    "mosaic"   : CFG["augmentation"]["mosaic"],
}

print("Training config:")
for k, v in TRAIN_CFG.items():
    print(f"  {k:20s}: {v}")


## 4.5 — Training Run

> **⚠️ Long-running cell.** Set `DRY_RUN = True` to validate config without training. Set `USE_PRETRAINED = False` to skip if you already have a checkpoint.

In [ ]:
DRY_RUN       = False   # ← set True to skip actual training
USE_PRETRAINED = True   # ← use a pre-trained checkpoint if it exists

pretrained_path = OUTPUT_DIR / "pill_detector" / "weights" / "best.pt"

if USE_PRETRAINED and pretrained_path.exists():
    print(f"✅  Pre-trained model found at: {pretrained_path}")
    print("   Skipping training. Delete the checkpoint or set USE_PRETRAINED=False to retrain.")
    model = YOLO(str(pretrained_path))

elif DRY_RUN:
    print("🚧  DRY_RUN=True — validating config only, no training.")
    model = YOLO(f"{CFG['model']['name']}.pt")
    print("Config validated. Set DRY_RUN=False to train.")

else:
    print(f"🚀  Starting training: {CFG['model']['name']} for {TRAIN_CFG['epochs']} epochs...")
    start = time.time()
    model = YOLO(f"{CFG['model']['name']}.pt")
    results = model.train(**TRAIN_CFG)
    elapsed = time.time() - start
    print(f"\n✅  Training complete in {elapsed/60:.1f} minutes.")
    print(f"Best checkpoint: {pretrained_path}")


## 4.6 — Training Curves

> Parse the YOLOv8 results CSV and plot loss + metric curves.

In [ ]:
results_csv = OUTPUT_DIR / "pill_detector" / "results.csv"

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()   # YOLOv8 sometimes pads column names
    print("Columns:", list(df.columns))
    print(df.tail(5))

    # Pick available columns
    loss_cols    = [c for c in df.columns if "loss" in c.lower()]
    metric_cols  = [c for c in df.columns if any(x in c.lower() for x in ["map", "precision", "recall"])]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training Curves", fontsize=13, fontweight="bold")

    for col in loss_cols:
        axes[0].plot(df["epoch"], df[col], label=col)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss"); axes[0].legend(fontsize=8)

    for col in metric_cols:
        axes[1].plot(df["epoch"], df[col], label=col)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
    axes[1].set_title("Metrics (mAP / Precision / Recall)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig("results/visualizations/training_curves.png", bbox_inches="tight")
    plt.show()
else:
    print("⚠️  results.csv not found — train the model first (Section 4.5).")
    print(f"   Expected path: {results_csv}")


## 4.7 — Save & Register Best Checkpoint

In [ ]:
checkpoint = OUTPUT_DIR / "pill_detector" / "weights" / "best.pt"

if checkpoint.exists():
    # Copy to a canonical path for downstream notebooks
    dest = OUTPUT_DIR / "best.pt"
    shutil.copy2(checkpoint, dest)
    print(f"✅  Best checkpoint registered at: {dest}")
    print(f"   Size: {dest.stat().st_size / 1e6:.1f} MB")
else:
    print(f"⚠️  Checkpoint not found at {checkpoint}")
    print("   Run Section 4.5 to train the model first.")


## ✅  Notebook 4 Complete
> Model trained and saved. Proceed to **Notebook 5 — Evaluation & Metrics**.